In [1]:
import sys

In [2]:
#!{sys.executable} -m pip install langchain-openai langchain-community
#!{sys.executable} -m pip install langchain-qdrant

In [3]:
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
#from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from qdrant_client.models import Distance, VectorParams
from qdrant_client import QdrantClient
from dotenv import load_dotenv
import pandas as pd

In [4]:
load_dotenv("si699.env")

True

## Part 1 Chunking

In [5]:
damage_df = pd.read_json("dnd5eapi_dump/damage-types.json")
magic_schools_df = pd.read_json("dnd5eapi_dump/magic-schools.json")
alignments_df = pd.read_json("dnd5eapi_dump/alignments.json")
skills_df = pd.read_json("dnd5eapi_dump/skills.json")
languages_df = pd.read_json("dnd5eapi_dump/languages.json")
weapon_df = pd.read_json("dnd5eapi_dump/weapon-properties.json")
conditions_df = pd.read_json("dnd5eapi_dump/conditions.json")

In [6]:
def create_chunk(endpoint, url, items):
    desc = items.get("desc", "")
    if isinstance(desc, list):
        desc = " ".join(desc)
    ability = items.get("ability_score", {}).get("name", None)
    text = f"""
Type: {endpoint.replace('-', ' ').title()}
Name: {items.get("name")}
Associated Ability: {ability}

Category: {items.get("type", None)}
Script: {items.get("script", None)}
Typical Speakers: {", ".join(items.get("typical_speakers", []))}

Description:
{desc}
""".strip()

    chunk = {
        "id": f"{endpoint}_{items['index']}",
        "text": text,
        "metadata": {
            "endpoint": endpoint,
            "name": items.get("name"),
            "index": items.get("index"),
            "url": url + items.get("index"),
            "updated_at": items.get("updated_at"),
            "category": endpoint
        }
    }

    return chunk

In [7]:
def build_chunks(df):
    chunks = []
    for i in range(len(df)):
        endpoint, url, items = df.iloc[i, :]
        chunk = create_chunk(endpoint, url, items)
        chunks.append(chunk)
    return chunks

In [8]:
chunks = build_chunks(damage_df)
chunks.extend(build_chunks(magic_schools_df))
chunks.extend(build_chunks(alignments_df))
chunks.extend(build_chunks(skills_df))
chunks.extend(build_chunks(languages_df))
chunks.extend(build_chunks(weapon_df))
chunks.extend(build_chunks(conditions_df))

## Part2: Embedding and Store Vectors

In [9]:
# model initialization
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
client = QdrantClient(path="./qdrant_local") # specify the path so that it persist locally on disk

docs = []
for chunk in chunks:
    docs.append(Document(
        page_content=chunk["text"],
        metadata = {
            **chunk["metadata"],
            "chunk_id": chunk["id"]
        }
    ))

In [11]:
collection_name = "damage_types"
client.delete_collection(collection_name="damage_types")
client.create_collection(
    collection_name= collection_name,
    vectors_config= VectorParams(size=1536, #dimensinality of the vector
                                 distance=Distance.COSINE # distance metric for similarity search
    )
)

vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embedding_model
)

vectorstore.add_documents(docs)


['b7099c010db946d6ac4718175d8e3a20',
 'b5c46231a9e446d489bb9ceb39ee3ff1',
 '108582fa1f6742c4b848829b8b9acb7a',
 '3378cd41a2204860a94a051993ad4ce4',
 'f48ed0c0ca64477fac834e964999a3b4',
 'a6e049e894f247ffa74586fae92ec81f',
 '476cff9f1527445d9b29f7f5ef0e31c9',
 'ea8930d03a31410982fcfcd1c8d9907f',
 '565be74fa8aa4663b4db315a0cbff215',
 '8fdb23e7e99d4dfead3476a383f3da1f',
 '99aedb27232141c5a73a7324988fbb08',
 '1f94b40848ed499a8ad6b9c2d7f8a31b',
 'a332e63cf6fe4e7da3d20ede2c6f70e4',
 '2eb44680e73c4166a4747eba4dbc83c8',
 'edd1eb48acd24d3f983960220fdcac39',
 'fda1245b8aa341d1a72ca562cab1ee8f',
 '6af58ed8a13d4fd1913ab1996fc77804',
 'ce434991736a4f658c82d887bdc99b42',
 '7388bf87cc324f25a690bb908cf796f6',
 'bb2854e42199450b80dfa2bf0e2c9498',
 '7a0520a6c48f4af686d71f67fcf312df',
 '1321e0a3c1f84b0e8eeeaa6d34595cb4',
 '846f0dbddf4f46fa9f2ef9b794a4bc65',
 '1ce4289accde404583f7ef05d01dfcc5',
 '2804be658a034700b6e5596039e68397',
 '0625eb7de9054a05be191e447f96322d',
 '2118fb637cbf4a2db92149f9a6111650',
 

## Part 3: Retrival

In [12]:
def retrieve_chunks(vectorstore, query, k=3):
    results = vectorstore.similarity_search(query, k=k)
    return results

In [13]:
def build_context(retrieved_docs):
    contexts = []
    for i, doc in enumerate(retrieved_docs, 1):
        contexts.append(f"[Chunk {i} | {doc.metadata.get('chunk_id')} | {doc.metadata.get('name')}]\n{doc.page_content}")
        
    return "\n\n".join(contexts)

In [14]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a helpful Dungeons & Dragons rules assistant.

Use only the retrieved context below to answer the question.
If the answer is not contained in the context, say "I don't know based on the retrieved context."

Retrieved Context:
{context}

Question:
{question}

Answer:
""".strip()
)

In [15]:
def rag_pipeline(query, llm, vectorstore, prompt_template, k=3):
    retrived_docs = retrieve_chunks(vectorstore, query, k)
    print(len(retrived_docs))
    context = build_context(retrived_docs)
    prompt = prompt_template.format(context=context, question=query)
    answer = llm.invoke(prompt).content
    
    return retrived_docs, context, answer

In [16]:
query = "If i want to fight with two weapons, what kind of weapon should i use?"

retrieved_docs, context, answer = rag_pipeline(
    query=query,
    vectorstore=vectorstore,
    llm=llm,
    prompt_template=rag_prompt,
    k=3
)

print(context)
print()
print(answer)

3
[Chunk 1 | weapon-properties_two-handed | Two-Handed]
Type: Weapon Properties
Name: Two-Handed
Associated Ability: None

Category: None
Script: None
Typical Speakers: 

Description:
This weapon requires two hands when you attack with it.

[Chunk 2 | weapon-properties_versatile | Versatile]
Type: Weapon Properties
Name: Versatile
Associated Ability: None

Category: None
Script: None
Typical Speakers: 

Description:
This weapon can be used with one or two hands. A damage value in parentheses appears with the property--the damage when the weapon is used with two hands to make a melee attack.

[Chunk 3 | weapon-properties_light | Light]
Type: Weapon Properties
Name: Light
Associated Ability: None

Category: None
Script: None
Typical Speakers: 

Description:
A light weapon is small and easy to handle, making it ideal for use when fighting with two weapons.

You should use light weapons, as they are small and easy to handle, making them ideal for fighting with two weapons.
